# 11 — ML Meta-Label Model Training

Amaç, final Robot stratejisinin ürettiği sinyalleri olasılık bazında
sıralayacak ilk meta-label modellerini karşılaştırmaktır.

Bu aşamada:

- Rastgele train-test split kullanılmaz.
- Development döneminde purged expanding-window cross-validation uygulanır.
- Bir fold'un eğitim etiketleri validation başlangıcından önce kapanmış olmalıdır.
- Model ve eşik adayları yalnızca Validation döneminde değerlendirilir.
- `Audit_2025_Plus` henüz modele gösterilmez.
- Nihai olasılık eşiği bu notebook'ta seçilmez; portföy backtestinde seçilir.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.ml_dataset import BASE_FEATURE_COLUMNS
from src.ml_training import (
    build_candidate_models,
    create_purged_expanding_folds,
    folds_summary,
    cross_validate_models,
    summarize_cv_metrics,
    train_on_development_predict_validation,
    event_threshold_table,
    validation_permutation_importance,
    save_candidate_model,
)


## 1. Meta-label eğitim veri setini yükle


In [ ]:
DATA_PATH = (
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

dataset = pd.read_parquet(DATA_PATH)

date_columns = [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]

for column in date_columns:
    dataset[column] = pd.to_datetime(
        dataset[column]
    )

development = (
    dataset.loc[
        dataset["Period"].eq("Development")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

validation = (
    dataset.loc[
        dataset["Period"].eq("Validation")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

audit = (
    dataset.loc[
        dataset["Period"].eq("Audit_2025_Plus")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

print("Development:", len(development))
print("Validation:", len(validation))
print("Audit — bu aşamada kullanılmayacak:", len(audit))
print("Özellik sayısı:", len(BASE_FEATURE_COLUMNS))


## 2. Sınıf dengesizliği ve rejim değişimi

Accuracy ana metrik değildir. Başarılı sinyal oranı yaklaşık üçte bir olduğu
için PR-AUC, precision lift, Brier ve olay getirisi birlikte incelenecektir.


In [ ]:
period_diagnostics = (
    dataset.groupby("Period")
    .agg(
        Event_Count=("Meta_Label", "size"),
        Positive_Rate=("Meta_Label", "mean"),
        Average_Return_=("Net_Return_%", "mean"),
        Median_Return_=("Net_Return_%", "median"),
        Average_R_Multiple=("R_Multiple", "mean"),
    )
    .reset_index()
)

period_diagnostics["Positive_Rate_%"] = (
    period_diagnostics["Positive_Rate"] * 100
)

display(period_diagnostics)


## 3. Purged expanding-window fold'ları oluştur


In [ ]:
folds = create_purged_expanding_folds(
    development_data=development,
    n_splits=4,
    initial_train_fraction=0.40,
    embargo_days=5,
)

fold_table = folds_summary(
    data=development,
    folds=folds,
    target_column="Meta_Label",
)

display(fold_table)

if not fold_table["Purged_Correctly"].all():
    raise RuntimeError(
        "Purged fold kontrolü başarısız."
    )


## 4. Aday modeller


In [ ]:
candidate_models = build_candidate_models(
    feature_columns=BASE_FEATURE_COLUMNS,
    random_state=42,
)

print("Modeller:")
for model_name in candidate_models:
    print("-", model_name)


## 5. Development purged cross-validation


In [ ]:
cv_fold_metrics, cv_oof_predictions = (
    cross_validate_models(
        development_data=development,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column="Meta_Label",
        models=candidate_models,
        folds=folds,
    )
)

cv_summary = summarize_cv_metrics(
    cv_fold_metrics
)

display(cv_summary)
display(
    cv_fold_metrics.sort_values(
        ["Model", "Fold"]
    )
)


### Metrik yorumu

- **PR-AUC:** Sınıf dengesizliğinde ana sıralama metriği.
- **ROC-AUC:** Genel ayrıştırma gücü.
- **Brier:** Olasılık kalitesi; düşük olması daha iyi.
- **Precision:** Modelin seçtiği sinyallerin başarı oranı.
- **Recall:** Başarılı sinyallerin ne kadarının yakalandığı.

PR-AUC mutlaka o dönemin pozitif sınıf oranıyla karşılaştırılmalıdır.


## 6. Development'ta eğit, Validation'ı tahmin et


In [ ]:
(
    fitted_models,
    validation_metrics,
    validation_predictions,
) = train_on_development_predict_validation(
    development_data=development,
    validation_data=validation,
    feature_columns=BASE_FEATURE_COLUMNS,
    target_column="Meta_Label",
    models=candidate_models,
    cutoff_date="2023-01-01",
    embargo_days=5,
)

display(validation_metrics)


## 7. Geçici model şampiyonunu belirle


In [ ]:
champion_name = (
    validation_metrics.sort_values(
        ["PR_AUC", "Brier", "ROC_AUC"],
        ascending=[False, True, False],
    )
    .iloc[0]["Model"]
)

champion_model = fitted_models[champion_name]

champion_validation = (
    validation_predictions.loc[
        validation_predictions["Model"].eq(
            champion_name
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("Geçici model şampiyonu:", champion_name)
print(
    "Validation pozitif sınıf baz oranı:",
    round(
        champion_validation["Meta_Label"].mean(),
        4,
    ),
)


Bu seçim yalnızca ilk aday model seçimidir. Nihai ML stratejisi sayılmamalıdır.
Bir sonraki adımda olasılık eşikleri event-driven portföy backtestine
entegre edilecektir.


## 8. Validation olasılık eşiklerinin olay kalitesi


In [ ]:
threshold_table = event_threshold_table(
    predictions=champion_validation,
    probability_column="Probability",
)

minimum_selected_count = max(
    100,
    int(len(champion_validation) * 0.20),
)

threshold_shortlist = (
    threshold_table.loc[
        threshold_table["Selected_Count"].ge(
            minimum_selected_count
        )
        & threshold_table["Precision_Lift_pp"].gt(0)
        & threshold_table[
            "Average_Return_Lift_pp"
        ].gt(0)
    ]
    .sort_values(
        [
            "Average_R_Multiple",
            "Profit_Factor",
            "Precision_Lift_pp",
        ],
        ascending=False,
    )
    .head(10)
    .reset_index(drop=True)
)

print("TÜM EŞİKLER")
display(threshold_table)

print("PORTFÖY BACKTESTİNE TAŞINACAK KISA LİSTE")
display(threshold_shortlist)


Eşik kısa listesi yalnızca aday üretir. En yüksek event getirili eşiği
doğrudan seçmek hatalı olabilir; yüksek eşik portföyü boş bırakabilir,
çeşitlendirmeyi azaltabilir veya işlem sıralamasını değiştirebilir.


## 9. Validation permutation importance


In [ ]:
permutation_table = (
    validation_permutation_importance(
        fitted_model=champion_model,
        validation_data=validation,
        feature_columns=BASE_FEATURE_COLUMNS,
        target_column="Meta_Label",
        scoring="average_precision",
        n_repeats=20,
        random_state=42,
    )
)

display(permutation_table.head(20))


In [ ]:
top_importance = permutation_table.head(15).sort_values(
    "Importance_Mean"
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_importance["Feature"],
    top_importance["Importance_Mean"],
)
plt.title(
    f"{champion_name} — Validation Permutation Importance"
)
plt.xlabel("PR-AUC önem azalması")
plt.ylabel("Özellik")
plt.tight_layout()
plt.show()


## 10. Sonuçları ve geçici modeli kaydet


In [ ]:
ML_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
)
ML_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

cv_fold_metrics.to_csv(
    ML_RESULTS_DIR
    / "meta_label_cv_fold_metrics.csv",
    index=False,
)

cv_summary.to_csv(
    ML_RESULTS_DIR
    / "meta_label_cv_summary.csv",
    index=False,
)

cv_oof_predictions.to_parquet(
    ML_RESULTS_DIR
    / "meta_label_development_oof_predictions.parquet",
    index=False,
)

validation_metrics.to_csv(
    ML_RESULTS_DIR
    / "meta_label_validation_metrics.csv",
    index=False,
)

validation_predictions.to_parquet(
    ML_RESULTS_DIR
    / "meta_label_validation_predictions.parquet",
    index=False,
)

threshold_table.to_csv(
    ML_RESULTS_DIR
    / "meta_label_threshold_table.csv",
    index=False,
)

threshold_shortlist.to_csv(
    ML_RESULTS_DIR
    / "meta_label_threshold_shortlist.csv",
    index=False,
)

permutation_table.to_csv(
    ML_RESULTS_DIR
    / "meta_label_permutation_importance.csv",
    index=False,
)

model_path, metadata_path = save_candidate_model(
    model=champion_model,
    model_name=champion_name,
    feature_columns=BASE_FEATURE_COLUMNS,
    output_directory=MODEL_DIR,
    training_end="2022-12-31",
    target_column="Meta_Label",
)

print("Model:", model_path)
print("Metadata:", metadata_path)
print(
    "Audit dönemi modele gösterilmedi:",
    len(audit),
    "olay",
)


## Sonraki aşama

Bir sonraki notebook:

1. Champion modelin validation olasılıklarını günlük Robot adaylarına bağlayacak.
2. Kısa listedeki olasılık eşiklerini event-driven portföy backtestinde çalıştıracak.
3. Her ML eşiğini orijinal Robot ve BIST100 ile karşılaştıracak.
4. İşlem sayısı, CAGR, drawdown, Profit Factor ve boşta kalma oranını birlikte değerlendirecek.
5. Eşik sabitlendikten sonra ilk kez `Audit_2025_Plus` raporu üretilecek.
